 ## Using LLMs for inference in zero-shot prompting
 This code was written mainly be me, however Claude was used to trouble shoot, specifically for the API timeout error that requires asyncio and the LengthFinishReasonError. Please refer to the LLM_labeling.ipynb notebook for specific comments on how the code functions as they share the same overall approach.
 
 The results displayed in this notebook are not the final outputs used in the thesis, they are older and use the default SEQEVAL metrics, which is not correct.

In [ ]:
from dotenv import load_dotenv
import os
import asyncio
import json
import os
import time
from openai import AsyncAzureOpenAI
from openai import AzureOpenAI
from openai import LengthFinishReasonError
from pydantic import BaseModel
from seqeval.metrics import classification_report
from pathlib import Path


In [2]:
load_dotenv(".env")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")

In [3]:
def get_list_of_sentences(file_path):
    """this returns a list of lists for sentences and list of lists for labels"""
    current_sent = []
    current_label = []
    all_sents = []
    all_labels = []

    with open(file_path, "r", encoding="utf-8") as infile:
        lines = infile.readlines()
        for line in lines:
            line = line.strip()
            if line == "":
                if len(current_sent) > 0:
                    all_sents.append(current_sent)
                    all_labels.append(current_label)
                    current_sent = []
                    current_label = []
            else:
                splitted = line.split("\t")
                token_part = splitted[0]
                label_part = splitted[1]
                current_sent.append(token_part)
                current_label.append(label_part)

        if len(current_sent) > 0:
            all_sents.append(current_sent)
            all_labels.append(current_label)
    
    return all_sents, all_labels


def results_classified(predicted_file, gold_file= r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll"):
    # gold_file = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_18th_March.conll"
    gold_sents, gold_labels = get_list_of_sentences(gold_file)
    llm_sents, llm_labels= get_list_of_sentences(predicted_file)
    y_true, y_pred = gold_labels, llm_labels
    report = classification_report(y_true, y_pred, digits=2)
    print(report)
    return
    
    # print(len(llm_sent_2), len(llm_labels_2))
    

In [ ]:
def ner_to_conll(token_map, entities):
    """this converts a token map + entity list to CoNLL BIO format."""
    entity_list = []
    new_list = []
    for thingies in entities:
        sent_dict = {}
        sentence_dict = {}
        sentence_dict["sentence_index"] = thingies.sent_index
        sentence_dict["entity_info"] = { "token": thingies.token_text,
                                        "token_index": thingies.token_index,
                                        "predicted_label": str(thingies.label).split(".")[1],
                                        "entity_index": thingies.entity_index}
        
        entity_list.append(sentence_dict)

    result = []
    prev_ent_index = None
    for dictionaries in token_map:
        prev_ent_index = None
        mapped_tokens = dictionaries["token_map"]
        if result:
             result.append("\n")
        for i in sorted(mapped_tokens.keys()):
            token = mapped_tokens[i]
            matched = False
            for entry in entity_list:
                    current_sent = dictionaries["sentence_number"]
                    if entry["sentence_index"] == current_sent and entry["entity_info"]["token_index"] == i:
                            label = entry["entity_info"]["predicted_label"]
                            ent_idx = entry["entity_info"]["entity_index"]
                            if ent_idx  == prev_ent_index:
                                prefix = "I"
                            else:
                                prefix = "B"
                                prev_ent_index = ent_idx
                            
                            matched = True
                            result.append((token, f"{prefix}-{label}"))
                            break
            if not matched:
                result.append((token, "O"))
                prev_ent_index = None
    return result

### Defining client

In [ ]:
def create_client() -> AsyncAzureOpenAI:
    return AsyncAzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version = "2024-12-01-preview"
    )

client = create_client()

model = "gpt-5.4-mini"
deployment ="gpt-5.4-mini"

# CREATING FINAL DATASET VARIABLE

In [ ]:
test_file = Path(r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")
tokens = []
labels = []
tokens_sentence = []
labels_sentence = []
token_count = []
with open(test_file, "r") as f:
    for line in f.readlines():
        if len(line) > 1:
            line = line.strip()
            token, label = line.split("\t")
            tokens_sentence.append(token)
            labels_sentence.append(label)
        else:
            if tokens_sentence:
                tokens.append(tokens_sentence)
                labels.append(labels_sentence)
                
            tokens_sentence = []
            labels_sentence = []
    if tokens_sentence:
        tokens.append(tokens_sentence)
        labels.append(labels_sentence)

print(len(tokens))

final_sents = []
counter = 0
for tok in tokens:
    new_sentence = " ".join(tok)
    final_sents.append([new_sentence])

with open("sentences.json", "w") as json_file:
    json.dump(final_sents, json_file)

656


In [ ]:
print(final_sents[0])

# GPT-NER ##TAGGING@@ <TECHNIQUE>

In [ ]:
def ent_dict_to_conll(entity_dictionary, sentence_list, output_file):
    """
    this writes the entity dictionary to conll with IOB2 labeling
    """
    with open(output_file, "w", encoding="utf-8") as outfile:
        for sent_index, sentence in enumerate(sentence_list):
            tokens = sentence.split(" ")
            
            sentence_entities = [
                e for e in entity_dictionary 
                if e.get("sentence_index") == sent_index
            ]

            for i, tok in enumerate(tokens):
                matched = False
                for entry in sentence_entities:
                    if entry["ent_start"] <= i <= entry["ent_end"]:
                        prefix = "B" if i == entry["ent_start"] else "I"
                        outfile.write(f"{tok}\t{prefix}-{entry['label']}\n")
                        matched = True
                        break
                if not matched:
                    outfile.write(f"{tok}\tO\n")

            outfile.write("\n")
                

In [ ]:
class Entity(BaseModel):
    sentence_index: int     ## sentence number
    full_sentence_with_markings: str        # exact surface form, e.g. "New York"

class NERResponse(BaseModel):
    entities: list[Entity]

def load_sentences(path):
    raw = json.load(open(path))
    return [group[0] for group in raw]

In [ ]:
## zero shot + simple descriptions
SYSTEM_PROMPT = """You are a Legal and Tax NER tagger. Given a sentence, return it exactly as provided but with every entity marked using the GPT-NER inline format:

    ##entity text@@ <LABEL>

where ## opens the entity span and @@ closes it, followed by a space, the label in angle brackets, and a space before the next token. Tokens that are not part of any entity are left unmarked. Never leave an entity span unclosed.

─── LABELS ───

PERSON
  Covers: people, including judges, advocates, parties, and named individuals.

ORG
  Covers: companies, agencies, institutions, and international bodies.
  Excludes: courts and tribunals (use COURT).

GPE
  Covers: specific named countries, cities, states, and regions.
  Excludes: abstract jurisdictional references (use JURISDICTION).

LAW
  Covers: named laws, treaties, directives, regulations, and conventions.
  Excludes: specific article or section references (use PROVISION).

DATE
  Covers: absolute dates, relative dates, and date ranges or periods.

TAX_TYPE
  Covers: named tax instruments as they appear in context.
  Excludes: jurisdictional modifiers preceding the instrument; generic nouns alone; verbal or metaphorical uses.

TAX_CONCEPT
  Covers: domain-specific terms that are not named tax instruments — legal doctrines, planning mechanisms, economic concepts, and behavioural descriptions.
  Excludes: accounting-standard vocabulary.

PROVISION
  Covers: specific articles, sections, paragraphs, or clauses within a law, including the law name when it appears as part of the reference.

JURISDICTION
  Covers: abstract or role-based jurisdictional references.
  Excludes: specific named countries or cities (use GPE).

COURT
  Covers: courts, tribunals, and judicial bodies, including generic references.

─── BOUNDARY RULES ───

1. Tag the minimal named span. Jurisdictional or descriptive modifiers before a TAX_TYPE are not part of the entity.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION.
3. Do not double-tag overlapping spans. Each token belongs to at most one entity.
4. Reproduce the original sentence exactly — do not alter, reorder, or omit any word. Only insert ## and @@ markers with their labels."""


In [ ]:
## zero shot - detailed descriptions WITHOUT EXAMPLES
SYSTEM_PROMPT = """You are a Legal and Tax NER tagger. Given a sentence, return it exactly as provided but with every entity marked using the GPT-NER inline format:

    ##entity text@@ <LABEL>

where ## opens the entity span and @@ closes it, followed by a space, the label in angle brackets, and a space before the next token. Tokens that are not part of any entity are left unmarked. Never leave an entity span unclosed.

─── LABELS ───

PERSON
  Covers: named or specifically referenced individuals in their human capacity, including judges, advocates general, named parties, and definite singular references resolvable to one specific individual in context. Spans include personal names with attached titles or roles. 
  Excludes: generic plurals, corporate or legal persons even when named like individuals, and unresolved generic role references.

ORG
  Covers: named non-judicial organisations — companies, agencies, supranational bodies, tax authorities, professional bodies). Spans include the full proper name with name-constitutive modifiers and standard legal-form suffixes. 
  Excludes: courts and tribunals (use COURT), jurisdictional modifiers that merely localise a generic institution, generic descriptions without naming , and entity-type concepts (use TAX_CONCEPT).

GPE
  Covers: specific, named geopolitical entities — countries, cities, regions, supranational unions with concrete identity. Spans cover the bare name; articles are included only when part of the official name. 
  Excludes: demonyms and adjectival forms entirely are never tagged on their own and are stripped when they modify another entity, abstract role-based references (use JURISDICTION), and geographic regions without political identity.

LAW
  Covers: named statutory or treaty-level instruments referred to as a whole, without article or section pinpointing. Spans include the full official title, standard acronyms, definite back-references that unambiguously resolve to a previously named instrument, and years embedded in a statutory title. 
  Excludes: pinpoint references (use PROVISION), case law and judgments, and generic legal-concept terms.

DATE
  Covers: absolute or relative temporal expressions denoting a point or period. Spans cover only the date expression itself; contextualizing nouns that label what kind of period it is are stripped. 
  Excludes: durations not anchored to a calendar point, vague temporal references, and dates embedded inside another entity's name.

TAX_TYPE: 
  Covers: named tax instruments as they appear in context.
  Exclude: jurisdictional modifiers, generic nouns alone, and verbal or metaphorical uses.

TAX_CONCEPT: 
  Covers: domain terms that aren't named instruments: legal doctrines, planning mechanisms, behaviours, and economic concepts.
  Excludes: accounting-standard vocabulary.

PROVISION
  Covers: specific articles, sections, paragraphs, or sub-paragraphs within a named instrument, including the host instrument when cited together. Spans cover the full citation as a single entity — pinpoint plus host instrument plus subdivisions — and coordinated pinpoints sharing one host as one span. 
  Excludes: the host instrument cited alone without a pinpoint (use LAW), case-paragraph references, and recital references unless project-scoped in.

JURISDICTION
  Covers: abstract or role-based references to a state in its legal or fiscal capacity, where the specific country is unnamed or generalized. Spans include the role descriptor with role-defining qualifiers. 
  Excludes: named countries even when filling such a role (use GPE) and purely geographic terms without legal-role meaning.

COURT
  Covers: judicial bodies, tribunals, and adjudicative offices. Spans include the full proper name with name-constitutive modifiers, standard acronyms, and case-specific unnamed references. 
  Excludes: jurisdictional modifiers that merely localize a generic court, individual judges or advocates general by name (use PERSON), and metaphorical uses.

─── BOUNDARY RULES ───

1. Tag the minimal named span. Jurisdictional or descriptive modifiers before a TAX_TYPE are not part of the entity.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION.
3. Do not double-tag overlapping spans. Each token belongs to at most one entity.
4. Reproduce the original sentence exactly — do not alter, reorder, or omit any word. Only insert ## and @@ markers with their labels."""


In [11]:
print(client)

In [ ]:
def get_ent_spans_from_string(marked_sentence, sentence_index):
    tokens = marked_sentence.split(" ")
    parsed = []
    inside = False
    real_index = 0
    current_span_tokens = []
    span_start = None

    for token in tokens:
        if token.startswith("##") and token.endswith("@@"):
            span_start = real_index
            current_span_tokens = [token[2:-2]]
            real_index += 1
        elif token.startswith("##"):
            span_start = real_index
            current_span_tokens = [token[2:]]
            inside = True
            real_index += 1
        elif inside and token.endswith("@@"):
            current_span_tokens.append(token[:-2])
            inside = False
            real_index += 1
        elif inside:
            current_span_tokens.append(token)
            real_index += 1
        elif token.startswith("<") and token.endswith(">") and current_span_tokens:
            parsed.append({
                "sentence_index": sentence_index,
                "ent_start": span_start,
                "ent_end": real_index - 1,
                "ent_text": " ".join(current_span_tokens),
                "label": token[1:-1],
            })
            current_span_tokens = []
            span_start = None
        else:
            real_index += 1

    if inside:
        print(f"Unclosed entity in sentence!!!: {sentence_index}: {marked_sentence}")

    return parsed

async def run_ner(client: AsyncAzureOpenAI, sentence: str, sentence_index: int) -> tuple[dict, dict]:
    try:
        response = await client.beta.chat.completions.parse(
            model="gpt-5.4-mini",
            max_completion_tokens=4096,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": sentence},
            ],
            response_format=NERResponse,
        )
        usage = {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }
        raw = {
            "sentence_index": sentence_index,
            "marked_sentences": [e.full_sentence_with_markings for e in response.choices[0].message.parsed.entities]
        }
        return raw, usage
    except LengthFinishReasonError as e:
        raise e

async def main():
    sentences = load_sentences("sentences.json")

    tasks = [run_ner(client, sentence, i) for i, sentence in enumerate(sentences)]
    results = await asyncio.gather(*tasks)

    raw_results = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for raw, usage in results:
        raw_results.append(raw)
        for key in total_usage:
            total_usage[key] += usage[key]

    with open("raw_marked_output.json", "w", encoding="utf-8") as f:
        json.dump(raw_results, f, indent=2, ensure_ascii=False)

    all_parsed_entities = []
    for raw in raw_results:
        sentence_index = raw["sentence_index"]
        for marked_sentence in raw["marked_sentences"]:
            parsed = get_ent_spans_from_string(marked_sentence, sentence_index)
            all_parsed_entities.extend(parsed)

    print(f"Found {len(all_parsed_entities)} entities")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    ent_dict_to_conll(all_parsed_entities, sentences, "gptner_zeroshot_1_may_gpt5.4.conll")
    return all_parsed_entities

In [ ]:
gptner_results = await main()
print(len(gptner_results))

### Results

In [ ]:
#new dataset -- gpt5.4-mini -- prompt 1
results_classified("gptner_zeroshot_1_may.conll", gold_file =  r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")

              precision    recall  f1-score   support

       COURT       0.33      0.68      0.44       106
        DATE       0.66      0.46      0.54       132
         GPE       0.65      0.60      0.62       299
JURISDICTION       0.46      0.19      0.27       153
         LAW       0.22      0.51      0.31        65
         ORG       0.38      0.33      0.36       221
      PERSON       0.71      0.79      0.75       186
   PROVISION       0.32      0.39      0.35        87
 TAX_CONCEPT       0.34      0.07      0.12       353
    TAX_TYPE       0.17      0.08      0.11        86

   micro avg       0.47      0.39      0.43      1688
   macro avg       0.43      0.41      0.39      1688
weighted avg       0.46      0.39      0.40      1688



In [ ]:
#new dataset -- gpt5.4-mini -- prompt 2 (old file)
results_classified("gptner_zeroshot_1_may_2_gpt5.4.conll", gold_file =  r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")

              precision    recall  f1-score   support

       COURT       0.39      0.55      0.46       106
        DATE       0.60      0.39      0.47       132
         GPE       0.65      0.60      0.62       299
JURISDICTION       0.37      0.12      0.18       153
         LAW       0.23      0.38      0.29        65
         ORG       0.47      0.33      0.39       221
      PERSON       0.72      0.62      0.67       186
   PROVISION       0.35      0.47      0.40        87
 TAX_CONCEPT       0.17      0.02      0.04       353
    TAX_TYPE       0.30      0.03      0.06        86

   micro avg       0.50      0.34      0.40      1688
   macro avg       0.43      0.35      0.36      1688
weighted avg       0.44      0.34      0.36      1688



# Index:Token Dictionary Marking

In [7]:
from enum import Enum
class Label(str, Enum):
    PERSON = 1
    ORG = 2
    GPE = 3
    LAW = 4
    DATE = 5
    TAX_TYPE = 6
    TAX_CONCEPT = 7
    PROVISION = 8
    JURISDICTION = 9
    COURT = 10

class Entity(BaseModel):
    token_text: str
    token_index: int   # key from the input token map
    label:  Label  # PER, LOC, or ORG #make e num -- to bound the values the variable can take
    entity_index: int      # the entity index so that it can be converted into B/I spans
    sent_index: int #sentence id

class NERResponse(BaseModel):
    entities: list[Entity]


In [ ]:
#simple descriptions + NO EXAMPLES 
SYSTEM_PROMPT = """You are a Legal and Tax NER classifier. Given a JSON object containing a sentence_number and a token_map (mapping token indices to tokens), identify all named entities and return a JSON array of entity tokens.

─── LABELS ───

PERSON
  Covers: people, including judges, advocates, parties, and named individuals.

ORG
  Covers: companies, agencies, institutions, and international bodies.
  Excludes: courts and tribunals (use COURT).

GPE
  Covers: specific named countries, cities, states, and regions.
  Excludes: abstract jurisdictional references (use JURISDICTION).

LAW
  Covers: named laws, treaties, directives, regulations, and conventions.
  Excludes: specific article or section references (use PROVISION).

DATE
  Covers: absolute dates, relative dates, and date ranges or periods.

TAX_TYPE
  Covers: named tax instruments as they appear in context.
  Excludes: jurisdictional modifiers preceding the instrument; generic nouns alone; verbal or metaphorical uses.

TAX_CONCEPT
  Covers: domain-specific terms that are not named tax instruments — legal doctrines, planning mechanisms, economic concepts, and behavioural descriptions.
  Excludes: accounting-standard vocabulary.

PROVISION
  Covers: specific articles, sections, paragraphs, or clauses within a law, including the law name when it appears as part of the reference.

JURISDICTION
  Covers: abstract or role-based jurisdictional references.
  Excludes: specific named countries or cities (use GPE).

COURT
  Covers: courts, tribunals, and judicial bodies, including generic references.

─── BOUNDARY RULES ───

1. Tag the minimal named span. Jurisdictional or descriptive modifiers before a TAX_TYPE are not part of the entity.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION.
3. Do not double-tag overlapping spans. Each token belongs to at most one entity.
4. Reproduce the original sentence exactly — do not alter, reorder, or omit any word. Only insert ## and @@ markers with their labels.

─── OUTPUT FORMAT ───

Return ONLY a JSON array. Each element represents one entity token:
{
  "sent_index": <sentence_number from input>,
  "token_index": <token position from token_map>,
  "token_text": <exact token string>,
  "entity_index": <integer grouping multi-token entities, starting at 0>,
  "label": "<one of the following integers representing the respective label: PERSON = 1, ORG = 2, GPE = 3, LAW = 4, DATE = 5, TAX_TYPE = 6, TAX_CONCEPT = 7, PROVISION = 8, JURISDICTION = 9, COURT = 10>"
}

Consecutive tokens sharing the same entity_index form one entity span. Increment entity_index for each new entity.
"""

In [ ]:
# detailed descriptions + NO EXAMPLES
SYSTEM_PROMPT= """You are a Legal and Tax NER classifier. Given a JSON object containing a sentence_number and a token_map (mapping token indices to tokens), identify all named entities and return a JSON array of entity tokens.

─── LABELS ───

PERSON
  Covers: named or specifically referenced individuals in their human capacity, including judges, advocates general, named parties, and definite singular references resolvable to one specific individual in context. Spans include personal names with attached titles or roles. 
  Excludes: generic plurals, corporate or legal persons even when named like individuals, and unresolved generic role references.

ORG
  Covers: named non-judicial organisations — companies, agencies, supranational bodies, tax authorities, professional bodies). Spans include the full proper name with name-constitutive modifiers and standard legal-form suffixes. 
  Excludes: courts and tribunals (use COURT), jurisdictional modifiers that merely localise a generic institution, generic descriptions without naming , and entity-type concepts (use TAX_CONCEPT).

GPE
  Covers: specific, named geopolitical entities — countries, cities, regions, supranational unions with concrete identity. Spans cover the bare name; articles are included only when part of the official name. 
  Excludes: demonyms and adjectival forms entirely are never tagged on their own and are stripped when they modify another entity, abstract role-based references (use JURISDICTION), and geographic regions without political identity.

LAW
  Covers: named statutory or treaty-level instruments referred to as a whole, without article or section pinpointing. Spans include the full official title, standard acronyms, definite back-references that unambiguously resolve to a previously named instrument, and years embedded in a statutory title. 
  Excludes: pinpoint references (use PROVISION), case law and judgments, and generic legal-concept terms.

DATE
  Covers: absolute or relative temporal expressions denoting a point or period. Spans cover only the date expression itself; contextualizing nouns that label what kind of period it is are stripped. 
  Excludes: durations not anchored to a calendar point, vague temporal references, and dates embedded inside another entity's name.

TAX_TYPE: 
  Covers: named tax instruments as they appear in context.
  Exclude: jurisdictional modifiers, generic nouns alone, and verbal or metaphorical uses.

TAX_CONCEPT: 
  Covers: domain terms that aren't named instruments: legal doctrines, planning mechanisms, behaviours, and economic concepts.
  Excludes: accounting-standard vocabulary.

PROVISION
  Covers: specific articles, sections, paragraphs, or sub-paragraphs within a named instrument, including the host instrument when cited together. Spans cover the full citation as a single entity — pinpoint plus host instrument plus subdivisions — and coordinated pinpoints sharing one host as one span. 
  Excludes: the host instrument cited alone without a pinpoint (use LAW), case-paragraph references, and recital references unless project-scoped in.

JURISDICTION
  Covers: abstract or role-based references to a state in its legal or fiscal capacity, where the specific country is unnamed or generalized. Spans include the role descriptor with role-defining qualifiers. 
  Excludes: named countries even when filling such a role (use GPE) and purely geographic terms without legal-role meaning.

COURT
  Covers: judicial bodies, tribunals, and adjudicative offices. Spans include the full proper name with name-constitutive modifiers, standard acronyms, and case-specific unnamed references. 
  Excludes: jurisdictional modifiers that merely localize a generic court, individual judges or advocates general by name (use PERSON), and metaphorical uses.

─── BOUNDARY RULES ───

1. Tag the minimal named span. Jurisdictional or descriptive modifiers before a TAX_TYPE are not part of the entity.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION.
3. Do not double-tag overlapping spans. Each token belongs to at most one entity.
4. Reproduce the original sentence exactly — do not alter, reorder, or omit any word.
─── OUTPUT FORMAT ───

Return ONLY a JSON array. Each element represents one entity token:
{
  "sent_index": <sentence_number from input>,
  "token_index": <token position from token_map>,
  "token_text": <exact token string>,
  "entity_index": <integer grouping multi-token entities, starting at 0>,
  "label": "<one of the following integers representing the respective label: PERSON = 1, ORG = 2, GPE = 3, LAW = 4, DATE = 5, TAX_TYPE = 6, TAX_CONCEPT = 7, PROVISION = 8, JURISDICTION = 9, COURT = 10>"
}

Consecutive tokens sharing the same entity_index form one entity span. Increment entity_index for each new entity.
"""

In [ ]:
def load_sentences(path):
    raw = json.load(open(path))
    return [group[0] for group in raw]


def build_token_map_lists(sentences):
    """this converts each sentence string into a numbered token map dict."""
    list_of_mappings = []
    for i, sentence in enumerate(sentences):
        tokens = sentence.split(" ")
        list_of_mappings.append({
            "sentence_number": i,
            "token_map": {j: token for j, token in enumerate(tokens)},
        })
    return list_of_mappings


async def run_ner(client, token_map):
    response = await client.beta.chat.completions.parse(
        model=deployment,
        temperature=0.0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(token_map)},
        ],
        response_format=NERResponse,
    )
    usage = {
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }
    return response.choices[0].message.parsed, usage


async def main():
    sentences = load_sentences("sentences.json")
    # sentences = sentences[0]
    token_maps = build_token_map_lists(sentences)

    client = create_client()
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities

In [ ]:
tokind_dictionary_zeroshot_1 = await main()
print(len(tokind_dictionary_zeroshot_1))

In [13]:
## get all token_maps
sentences = load_sentences("sentences.json")
token_maps = build_token_map_lists(sentences)

In [14]:
results = ner_to_conll(token_maps, tokind_dictionary_zeroshot_1)
with open("dictionary_zeroshot_1_may_2.conll", "w", encoding ="utf-8") as outfile:
    for element in results:
        if len(element) == 2:
             t, l = element
             outfile.write(f"{t}\t{l}\n")
        else:
             outfile.write(element)

In [ ]:
#new dataset -- gpt5.4-mini -- prompt 1
results_classified("dictionary_zeroshot_1_may.conll")

              precision    recall  f1-score   support

       COURT       0.26      0.76      0.39       106
        DATE       0.76      0.73      0.75       132
         GPE       0.56      0.69      0.62       299
JURISDICTION       0.26      0.37      0.30       153
         LAW       0.19      0.46      0.27        65
         ORG       0.38      0.32      0.35       221
      PERSON       0.61      0.74      0.67       186
   PROVISION       0.24      0.54      0.33        87
 TAX_CONCEPT       0.24      0.22      0.23       353
    TAX_TYPE       0.26      0.36      0.31        86

   micro avg       0.37      0.49      0.43      1688
   macro avg       0.38      0.52      0.42      1688
weighted avg       0.40      0.49      0.43      1688



In [ ]:
#new dataset -- gpt5.4-mini -- prompt 2
results_classified("dictionary_zeroshot_1_may_2.conll")

              precision    recall  f1-score   support

       COURT       0.32      0.73      0.45       106
        DATE       0.71      0.64      0.67       132
         GPE       0.61      0.78      0.68       299
JURISDICTION       0.17      0.14      0.15       153
         LAW       0.19      0.43      0.26        65
         ORG       0.41      0.38      0.39       221
      PERSON       0.68      0.67      0.68       186
   PROVISION       0.31      0.57      0.40        87
 TAX_CONCEPT       0.21      0.10      0.13       353
    TAX_TYPE       0.19      0.21      0.20        86

   micro avg       0.41      0.45      0.43      1688
   macro avg       0.38      0.46      0.40      1688
weighted avg       0.40      0.45      0.41      1688

